In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# GARCH with 80/20 train/test split

For 80% training rows:
- reconstruct chronological time series
- fit one StandardScaler on training series
- fit one GARCH(1,1) on training series
- save fitted parameters

For every X window in test series:
- transform X using TRAIN scaler
- apply the TRAINED GARCH parameters to that history
- forecast 20 steps
- inverse-transform forecast
- calculate CRPS against test Y

In [ ]:
import arch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from arch import arch_model
from scipy.stats import norm

QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS
):
    """
    y:
        shape (...,)

    quantile_forecasts:
        shape (..., Q)

    returns:
        CRPS approximation with shape (...)
    """

    y = np.asarray(y, dtype=float)
    qf = np.asarray(quantile_forecasts, dtype=float)
    levels = np.asarray(quantile_levels, dtype=float)

    error = y[..., None] - qf

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error
    )

    return 2.0 * np.mean(pinball, axis=-1)

def fit_garch_11_on_train(train_series):
    """
    Fit StandardScaler + GARCH(1,1) ONCE using the training series only.
    """

    train_series = np.asarray(
        train_series,
        dtype=float
    )

    # -----------------------------------------
    # 1. Fit scaler on TRAINING DATA ONLY
    # -----------------------------------------
    scaler = StandardScaler()

    train_std = scaler.fit_transform(
        train_series.reshape(-1, 1)
    ).ravel()

    # -----------------------------------------
    # 2. Fit GARCH(1,1) on standardized train
    # -----------------------------------------
    model = arch_model(
        train_std,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=False
    )

    result = model.fit(
        disp="off"
    )

    return result, scaler

def forecast_garch_11_fixed(
    x,
    trained_result,
    scaler,
    horizon=20,
    quantile_levels=QUANTILE_LEVELS
):
    """
    Forecast one test context using the GARCH parameters
    learned from the training set.

    NO refitting is performed on the test window.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    # -----------------------------------------
    # 1. Transform test X using TRAIN scaler
    # -----------------------------------------
    x_std = scaler.transform(
        x.reshape(-1, 1)
    ).ravel()

    # -----------------------------------------
    # 2. Create same GARCH structure on this
    #    observed history
    # -----------------------------------------
    test_model = arch_model(
        x_std,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=False
    )

    # -----------------------------------------
    # 3. FIX parameters at values learned
    #    from training set
    # -----------------------------------------
    fixed_result = test_model.fix(
        trained_result.params
    )

    # -----------------------------------------
    # 4. Forecast 20 steps
    # -----------------------------------------
    forecast = fixed_result.forecast(
        horizon=horizon,
        reindex=False
    )

    mu_std = (
        forecast.mean
        .iloc[-1]
        .to_numpy(dtype=float)
    )

    variance_std = (
        forecast.variance
        .iloc[-1]
        .to_numpy(dtype=float)
    )

    variance_std = np.maximum(
        variance_std,
        1e-12
    )

    sigma_std = np.sqrt(
        variance_std
    )

    # -----------------------------------------
    # 5. Gaussian quantiles
    # -----------------------------------------
    z = norm.ppf(
        quantile_levels
    )

    quantiles_std = (
        mu_std[:, None]
        + sigma_std[:, None] * z[None, :]
    )

    # -----------------------------------------
    # 6. Return to original scale
    # -----------------------------------------
    scale = scaler.scale_[0]
    center = scaler.mean_[0]

    mu = (
        mu_std * scale
        + center
    )

    sigma = (
        sigma_std * scale
    )

    quantiles = (
        quantiles_std * scale
        + center
    )

    return mu, sigma, quantiles

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

SOURCE_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")
RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/garch_standardized_train_test")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 250
HORIZON = 20
P = 1
Q = 1

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

summary_rows = []


# ============================================================
# LOOP OVER ASSET CLASSES
# ============================================================

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = SOURCE_DIR / folder_name
    output_dir = RESULTS_DIR / folder_name

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    if not input_dir.exists():
        print(f"Missing folder: {input_dir}")
        continue

    train_files = sorted(
        input_dir.glob(
            "train_*_no_preprocess_"
            "lookback-250_horizon-20_slide-1.csv"
        )
    )

    print("\n" + "=" * 80)
    print(f"{asset_class}: {len(train_files)} datasets")
    print("=" * 80)

    # ========================================================
    # LOOP OVER DATASETS
    # ========================================================

    for train_csv in train_files:

        suffix = (
            "_no_preprocess_"
            "lookback-250_"
            "horizon-20_"
            "slide-1.csv"
        )

        dataset_name = (
            train_csv.name
            .removeprefix("train_")
            .removesuffix(suffix)
        )

        # Keep Tether excluded
        if "tether" in dataset_name.lower():
            print(f"Skipping Tether: {dataset_name}")
            continue

        test_csv = input_dir / (
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1.csv"
        )

        if not test_csv.exists():
            print(f"Missing test file: {test_csv}")
            continue

        print("\n" + "-" * 70)
        print(f"Dataset: {dataset_name}")
        print(f"Asset class: {asset_class}")
        print("-" * 70)

        # ====================================================
        # LOAD TRAIN + TEST
        # ====================================================

        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv)

        x_cols = [
            f"X_{i}"
            for i in range(LOOKBACK)
        ]

        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]

        X_train = train_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_train = train_df[
            y_cols
        ].to_numpy(dtype=float)

        X_test = test_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_test = test_df[
            y_cols
        ].to_numpy(dtype=float)

        # ====================================================
        # BASIC CHECKS
        # ====================================================

        if len(X_train) == 0:
            print("Skipping: empty training set")
            continue

        if len(X_test) == 0:
            print("Skipping: empty test set")
            continue

        if not np.isfinite(X_train).all():
            print("Skipping: non-finite X_train")
            continue

        if not np.isfinite(Y_train).all():
            print("Skipping: non-finite Y_train")
            continue

        if not np.isfinite(X_test).all():
            print("Skipping: non-finite X_test")
            continue

        if not np.isfinite(Y_test).all():
            print("Skipping: non-finite Y_test")
            continue

        # ====================================================
        # RECONSTRUCT CHRONOLOGICAL TRAINING SERIES
        # slide = 1
        # ====================================================

        train_series = list(
            X_train[0]
        )

        for i in range(1, len(X_train)):
            train_series.extend(
                X_train[i, -1:]
            )

        # Include final known training future
        train_series.extend(
            Y_train[-1]
        )

        train_series = np.asarray(
            train_series,
            dtype=float
        )

        print(
            "Train rows:",
            len(X_train)
        )

        print(
            "Test rows:",
            len(X_test)
        )

        print(
            "Reconstructed train series:",
            len(train_series)
        )

        # ====================================================
        # FIT GARCH(1,1) ONCE
        # ====================================================

        try:

            trained_garch, train_scaler = (
                fit_garch_11_on_train(
                    train_series
                )
            )

        except Exception as e:

            print(
                f"Training failed for "
                f"{dataset_name}: {e}"
            )

            continue

        print(
            "GARCH training complete."
        )

        print(
            trained_garch.params
        )

        # ====================================================
        # STORAGE
        # ====================================================

        N = len(X_test)
        H = HORIZON
        NQ = len(QUANTILE_LEVELS)

        garch_test_quantiles = np.full(
            (N, H, NQ),
            np.nan,
            dtype=float
        )

        garch_test_crps = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        point_forecasts = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        failed_windows = []

        # ====================================================
        # TEST LOOP
        # ====================================================

        for i in range(N):

            try:

                (
                    mu,
                    sigma,
                    quantiles
                ) = forecast_garch_11_fixed(
                    x=X_test[i],
                    trained_result=trained_garch,
                    scaler=train_scaler,
                    horizon=H,
                    quantile_levels=QUANTILE_LEVELS
                )

                crps_i = quantile_crps(
                    Y_test[i],
                    quantiles,
                    QUANTILE_LEVELS
                )

                if not np.all(
                    np.isfinite(crps_i)
                ):
                    raise RuntimeError(
                        "Non-finite qCRPS"
                    )

                point_forecasts[i] = mu

                garch_test_quantiles[i] = (
                    quantiles
                )

                garch_test_crps[i] = (
                    crps_i
                )

            except Exception as e:

                failed_windows.append({
                    "window": i,
                    "error": str(e)
                })

            if (i + 1) % 100 == 0:
                print(
                    f"{dataset_name}: "
                    f"{i + 1}/{N}"
                )

        # ====================================================
        # qCRPS
        # ====================================================

        crps_by_horizon = np.nanmean(
            garch_test_crps,
            axis=0
        )

        mean_qcrps = np.nanmean(
            crps_by_horizon
        )

        n_failed = len(
            failed_windows
        )

        n_valid = (
            N - n_failed
        )

        print(
            f"Mean qCRPS: "
            f"{mean_qcrps:.8f}"
        )

        print(
            f"Valid windows: "
            f"{n_valid}/{N}"
        )

        # ====================================================
        # FILE PREFIX
        # ====================================================

        clean_name = (
            dataset_name
            .lower()
            .replace(
                " historical data",
                ""
            )
            .replace(
                " ",
                "_"
            )
            .replace(
                ".",
                ""
            )
        )

        prefix = (
            f"{folder_name.lower()}_"
            f"{clean_name}_"
            f"garch_1_1_train_test"
        )

        # ====================================================
        # SAVE QUANTILES
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"{prefix}_quantiles.npz",

            quantiles=(
                garch_test_quantiles
            ),

            quantile_levels=(
                QUANTILE_LEVELS
            )
        )

        # ====================================================
        # SAVE FULL CRPS ARRAY
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"{prefix}_crps.npz",

            crps=(
                garch_test_crps
            )
        )

        # ====================================================
        # SAVE qCRPS BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                H + 1
            ),
            "mean_qcrps": (
                crps_by_horizon
            )
        }).to_csv(
            output_dir /
            f"{prefix}_crps_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SAVE POINT FORECAST CSV
        # ====================================================

        point_df = test_df.copy()

        for h, col in enumerate(y_cols):
            point_df[col] = (
                point_forecasts[:, h]
            )

        point_df.to_csv(
            output_dir /
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1_"
            f"garch-1-1.csv",
            index=False
        )

        # ====================================================
        # SAVE FAILURES
        # ====================================================

        if failed_windows:

            pd.DataFrame(
                failed_windows
            ).to_csv(
                output_dir /
                f"{prefix}_failed_windows.csv",
                index=False
            )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class": (
                asset_class
            ),
            "dataset": (
                dataset_name
            ),
            "model": (
                "GARCH(1,1)"
            ),
            "mean_qcrps": (
                mean_qcrps
            ),
            "n_train_windows": (
                len(X_train)
            ),
            "n_test_windows": (
                N
            ),
            "n_valid": (
                n_valid
            ),
            "n_failed": (
                n_failed
            ),
            "failure_rate": (
                n_failed / N
            )
        })


# ============================================================
# SAVE SUMMARY FOR ALL DATASETS
# ============================================================

garch_summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULTS_DIR /
    "garch_1_1_train_test_"
    "summary_all_datasets.csv"
)

garch_summary.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Summary saved to:\n"
    f"{summary_path}"
)

display(
    garch_summary
)